# PathSnap - live training re-run

Re-trains the candidate classifier from the repository code on the provided data and checks that the results match the ones committed in the repository (`outputs/`). Training is seeded (42), so the metrics should be identical; only timings differ.

**Run the cells in order with Shift+Enter.** You need the dataset zip (`Student_folder.zip` / `Part2_dataset_Student_folder.zip`).

## 1. Get the code and install the pinned packages (2-4 min)
If Colab then asks to **restart the session**, click it and continue with step 2 (the next cells change into the project folder themselves).

In [ ]:
%cd /content
!test -d ML_Hackathon || git clone https://github.com/Ansh7-7/ML_Hackathon
%cd /content/ML_Hackathon
!git log --oneline | head -3
!pip install -q -r requirements.txt

## 2. Upload the dataset zip
A file picker appears below - choose the dataset zip.

In [ ]:
%cd /content/ML_Hackathon
from google.colab import files
up = files.upload()

## 3. Put the data in place

In [ ]:
%cd /content/ML_Hackathon
import zipfile, shutil, os
z = list(up)[0]
zipfile.ZipFile(z).extractall('data/tmp')
src = 'data/tmp/Student folder'
os.makedirs('data/raw', exist_ok=True)
for d in ['train', 'student_test']:
    if not os.path.exists(f'data/raw/{d}'):
        shutil.move(f'{src}/{d}', f'data/raw/{d}')
print({d: len(os.listdir(f'data/raw/{d}')) for d in ['train', 'student_test']}, 'trips')

## 4. Candidate generation + feature engineering (~1 min)

In [ ]:
%cd /content/ML_Hackathon
!python scripts/02_build_dataset.py | tail -8

## 5. Train and evaluate XGBoost (~2 min)
5-fold GroupKFold cross-validation by trip on the 78 train trips, then a final fit evaluated on the 10 student-test trips.

In [ ]:
%cd /content/ML_Hackathon
!python scripts/03_train_compare.py --models XGBoost --force

## 6. Compare with the results committed in the repository

In [ ]:
%cd /content/ML_Hackathon
import json, subprocess
new = json.load(open('outputs/models/per_model/XGBoost.json'))
old = json.loads(subprocess.run(['git', 'show', 'HEAD:outputs/models/per_model/XGBoost.json'], capture_output=True, text=True).stdout)
ok = True
for k in ['cv_top1_mean', 'cv_macro_f1_mean', 'cv_log_loss_mean', 'st_top1', 'st_macro_f1', 'st_log_loss']:
    m = abs(old[k] - new[k]) < 5e-4
    ok &= m
    print(f'{k:18s} repo={old[k]:.4f}  colab={new[k]:.4f}  match={m}')
print('\nALL MATCH' if ok else '\nsome values differ - see above')

## 7. Optional - more models (~5 min)

In [ ]:
%cd /content/ML_Hackathon
!python scripts/03_train_compare.py --models LogisticRegression GaussianNB LightGBM MLP --force

## 8. Optional - a short Bayesian-optimisation run (~6-8 min, 5 trials)
The submitted run used 40 trials (`outputs/hpo/trials.csv`). This only changes this Colab copy, never the GitHub repository.

In [ ]:
%cd /content/ML_Hackathon
!python scripts/04_bayes_opt.py --model XGBoost --trials 5 2>&1 | grep -v Warning | tail -12